Assembly line balancing problems can be formulated using a mathematical model based on linear programming. This approach, proposed by E. H. Bowman in 1960, is used to balance workloads across workstations on an assembly line. The model aims to distribute tasks in the most efficient way possible under a given assembly line configuration.üzeni altında, işlerin en uygun şekilde dağıtılmasını amaçlar.  gpptye bunu yazmıştım, formülasyonları getirmişti.

In [1]:
from gurobipy import Model, GRB, quicksum

# Task durations (in minutes)
tasks = {
    1: 5, 2: 3, 3: 4, 4: 3, 5: 2, 6: 2,
    7: 4, 8: 3, 9: 6, 10: 5, 11: 4, 12: 2
}

# Precedence relationships (task1, task2) => task1 must be completed before task2
precedence = [
    (1, 3), (2, 5), (5, 6), (3, 7), (4, 8), (8, 9)
]

# Cycle time
C = 10

# Let's estimate the total number of stations (up to the maximum number of tasks).
max_stations = len(tasks)

In [2]:
from gurobipy import Model, GRB, quicksum
import numpy as np

# Function to generate random assembly line data
def generate_random_assembly_data(num_tasks=20, num_precedence=10, cycle_time_range=(10, 20)):
    # Random task durations (between 1 and 10 minutes)
    tasks = {i: np.random.randint(1, 11) for i in range(1, num_tasks + 1)}
    
    # Random precedence relationships
    precedence = []
    while len(precedence) < num_precedence:
        task1, task2 = np.random.choice(list(tasks.keys()), 2, replace=False)
        if (task1, task2) not in precedence and task1 != task2:
            precedence.append((task1, task2))
    
    # Random cycle time (between 10 and 20)
    cycle_time = np.random.randint(*cycle_time_range)
    
    return tasks, precedence, cycle_time

# Generate random data
tasks, precedence, C = generate_random_assembly_data(num_tasks=100, num_precedence=50, cycle_time_range=(10, 20))

# Estimate total number of stations (up to the number of tasks)
max_stations = len(tasks)

# Total task time
total_task_time = sum(tasks.values())

# Minimum theoretical number of stations
min_stations = np.ceil(total_task_time / C)

# Print results
print("Randomly generated tasks (task durations):", tasks)
print("\nRandomly generated precedence relationships:", precedence)
print("\nRandomly generated cycle time (C):", C)
print(f"\nTotal task time: {total_task_time} minutes")
print(f"Minimum theoretical number of stations: {int(min_stations)}")

# Estimate total number of stations (slightly more than theoretical minimum)
max_stations = int(min_stations) + 4

Randomly generated tasks (task durations): {1: 7, 2: 5, 3: 6, 4: 5, 5: 4, 6: 4, 7: 6, 8: 8, 9: 4, 10: 6, 11: 1, 12: 8, 13: 7, 14: 3, 15: 6, 16: 10, 17: 10, 18: 9, 19: 9, 20: 1, 21: 4, 22: 8, 23: 5, 24: 5, 25: 1, 26: 2, 27: 7, 28: 4, 29: 1, 30: 7, 31: 5, 32: 6, 33: 7, 34: 9, 35: 1, 36: 8, 37: 6, 38: 4, 39: 10, 40: 7, 41: 2, 42: 7, 43: 6, 44: 3, 45: 1, 46: 9, 47: 6, 48: 2, 49: 8, 50: 4, 51: 9, 52: 1, 53: 9, 54: 10, 55: 10, 56: 5, 57: 8, 58: 8, 59: 7, 60: 6, 61: 4, 62: 6, 63: 1, 64: 6, 65: 2, 66: 5, 67: 2, 68: 4, 69: 8, 70: 2, 71: 7, 72: 3, 73: 2, 74: 6, 75: 8, 76: 4, 77: 8, 78: 4, 79: 6, 80: 9, 81: 5, 82: 10, 83: 1, 84: 8, 85: 2, 86: 3, 87: 6, 88: 7, 89: 9, 90: 2, 91: 3, 92: 6, 93: 5, 94: 8, 95: 6, 96: 2, 97: 3, 98: 5, 99: 2, 100: 1}

Randomly generated precedence relationships: [(12, 36), (81, 27), (22, 82), (16, 56), (92, 56), (42, 52), (46, 24), (73, 70), (30, 81), (26, 92), (82, 90), (65, 93), (68, 55), (19, 31), (62, 69), (39, 10), (41, 84), (25, 15), (35, 95), (66, 10), (9, 49), (9

In [3]:
# Create Gurobi model
model = Model("Assembly_Line_Balancing")

# Variables
# x[i, j] = Is task i assigned to station j?
x = model.addVars(tasks.keys(), range(1, max_stations + 1), vtype=GRB.BINARY, name="x")

# y[j] = Is station j used?
y = model.addVars(range(1, max_stations + 1), vtype=GRB.BINARY, name="y")

# Objective: Minimize the total number of stations used
model.setObjective(quicksum(y[j] for j in range(1, max_stations + 1)), GRB.MINIMIZE)

# If a task is assigned to a station, that station must be marked as used
for i in tasks.keys():
    for j in range(1, max_stations + 1):
        model.addConstr(x[i, j] <= y[j], f"StationUsage_{i}_{j}")

# Constraints

# Each task must be assigned to exactly one station
for i in tasks.keys():
    model.addConstr(quicksum(x[i, j] for j in range(1, max_stations + 1)) == 1, f"AssignTask_{i}")

# The total time of tasks assigned to a station must not exceed the cycle time
for j in range(1, max_stations + 1):
    model.addConstr(
        quicksum(tasks[i] * x[i, j] for i in tasks.keys()) <= C,
        f"CycleTime_Station_{j}"
    )

# Respect precedence constraints
for i, k in precedence:  # i must be before k
    for j in range(1, max_stations + 1):
        model.addConstr(
            quicksum(x[i, jj] for jj in range(1, j + 1)) >= x[k, j],
            f"Precedence_{i}_{k}_Station_{j}"
        )

# Optional: Immediate precedence (tighter constraints, can be enabled if needed)
# for i, k in precedence:
#     for j in range(1, max_stations + 1):
#         if j <= 2:
#             model.addConstr(
#                 quicksum(x[i, jj] for jj in range(1, j + 1)) >= x[k, j],
#                 f"Immediate_Precedence_{i}_{k}_Station_{j}"
#             )
#         else:
#             model.addConstr(
#                 quicksum(x[i, jj] for jj in range(j - 1, j + 1)) >= x[k, j],
#                 f"Immediate_Precedence_{i}_{k}_Station_{j}"
#             )

# Symmetry breaking: enforce ordered usage of stations
for j in range(1, max_stations):
    model.addConstr(y[j] >= y[j + 1], f"Symmetry_{j}")

Set parameter Username
Academic license - for non-commercial use only - expires 2025-08-29


In [4]:
# Optimize the model
model.optimize()

# Print the results
if model.status == GRB.OPTIMAL:
    print("\nOptimal solution found:")
    for j in range(1, max_stations + 1):
        # Tasks assigned to station j
        assigned_tasks = [i for i in tasks.keys() if x[i, j].x > 0.5]
        if assigned_tasks:
            # Total load of station j (in minutes)
            station_load = sum(tasks[i] for i in assigned_tasks)
            # Load percentage = total load / cycle time * 100
            percentage_load = (station_load / C) * 100
            print(f"Station {j}: Tasks {assigned_tasks}, Load: {station_load} minutes ({percentage_load:.2f}%)")
    print(f"\nTotal number of stations used: {int(model.objVal)}")
else:
    print("No optimal solution found!")

Gurobi Optimizer version 10.0.0 build v10.0.0rc2 (win64)

CPU model: Intel(R) Core(TM) i7-3630QM CPU @ 2.40GHz, instruction set [SSE2|AVX]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 6635 rows, 4343 columns and 66734 nonzeros
Model fingerprint: 0x96f6a4d1
Variable types: 0 continuous, 4343 integer (4343 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+01]
Found heuristic solution: objective 43.0000000
Presolve removed 1268 rows and 2 columns
Presolve time: 0.41s
Presolved: 5367 rows, 4341 columns, 76424 nonzeros
Variable types: 0 continuous, 4341 integer (4341 binary)

Root relaxation: objective 2.289000e+01, 3530 iterations, 0.50 seconds (0.60 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 